# Synthetic large image tutorial

This notebook demonstrates the non-georeferenced `large_images` workflow using pixel-coordinate COCO annotations. It creates a deterministic synthetic image, splits it into overlapping tiles, writes tile pickles, registers tile predictions back into global image coordinates, and evaluates the merged output.

In [1]:
import json
import shutil
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "jupyter_notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from instance_mosaicking.large_images.ellipse_instance_generation import generate_fixture
from instance_mosaicking.large_images.workflow import Workflow
from instance_mosaicking.large_images.evaluation import evaluate

output_root = repo_root / "jupyter_notebooks" / "synthetic_large_image_tutorial_output"
if output_root.exists():
    shutil.rmtree(output_root)
input_dir = output_root / "input"
workflow_dir = output_root / "workflow"
input_dir.mkdir(parents=True)

print(f"Repository root: {repo_root}")
print(f"Tutorial output: {output_root}")

Repository root: C:\Users\zhiang\Documents\GitHub\instance_mosaicking
Tutorial output: C:\Users\zhiang\Documents\GitHub\instance_mosaicking\jupyter_notebooks\synthetic_large_image_tutorial_output


In [2]:
ellipse_count = 12
fixture = generate_fixture(input_dir, image_size=(384, 384), ellipse_count=ellipse_count)
image_path = Path(fixture["image_path"])
annotation_path = Path(fixture["annotation_path"])
annotations = json.loads(annotation_path.read_text(encoding="utf-8"))

print(f"Input image: {image_path}")
print(f"Input annotations: {annotation_path}")
print(f"Image size: {annotations['images'][0]['width']} x {annotations['images'][0]['height']}")
print(f"Annotation count: {len(annotations['annotations'])}")
assert image_path.exists()
assert annotation_path.exists()
assert len(annotations["annotations"]) == ellipse_count

Input image: C:\Users\zhiang\Documents\GitHub\instance_mosaicking\jupyter_notebooks\synthetic_large_image_tutorial_output\input\large_image_fixture.png
Input annotations: C:\Users\zhiang\Documents\GitHub\instance_mosaicking\jupyter_notebooks\synthetic_large_image_tutorial_output\input\large_image_fixture.json
Image size: 384 x 384
Annotation count: 12


In [3]:
workflow = Workflow(
    image_path=image_path,
    annotation_path=annotation_path,
    save_dir=workflow_dir,
    tile_size=160,
    overlap=40,
    keep_empty_tiles=True,
    detection_threshold=0.0,
    iou_threshold=0.4,
)

metadata = workflow.split()
tile_names = [tile["name"] for tile in metadata["tiles"]]

print(f"Tile count: {len(tile_names)}")
print(f"First three tiles: {tile_names[:3]}")
assert len(tile_names) == 9
assert (workflow_dir / "tiles.json").exists()
assert (workflow_dir / "tile_annotations.json").exists()

Tile count: 9
First three tiles: ['0_0.png', '1_0.png', '2_0.png']


In [4]:
pickle_paths = workflow.save_training_pickles()

print(f"Pickle count: {len(pickle_paths)}")
print(f"First pickle: {pickle_paths[0]}")
assert len(pickle_paths) == 9
assert all(Path(path).exists() for path in pickle_paths)

Pickle count: 9
First pickle: C:\Users\zhiang\Documents\GitHub\instance_mosaicking\jupyter_notebooks\synthetic_large_image_tutorial_output\workflow\0_0.pickle


In [5]:
updated_files, timestamps = workflow.register_predictions(workflow_dir)
merged_path = workflow_dir / "merged_instances.json"
pixel_polygons_path = workflow_dir / "merged_pixel_polygons.json"
mask_path = workflow_dir / "merged_instance_mask.png"
overlay_path = workflow_dir / "merged_overlay.png"

merged = json.loads(merged_path.read_text(encoding="utf-8"))

print(f"Registered prediction files: {len(updated_files)}")
print(f"Merged instances: {len(merged['annotations'])}")
print(f"Merged COCO JSON: {merged_path}")
print(f"Pixel polygon JSON: {pixel_polygons_path}")
print(f"Instance mask PNG: {mask_path}")
print(f"Overlay PNG: {overlay_path}")

assert len(merged["annotations"]) == ellipse_count
assert pixel_polygons_path.exists()
assert mask_path.exists()
assert overlay_path.exists()

Registered prediction files: 9
Merged instances: 12
Merged COCO JSON: C:\Users\zhiang\Documents\GitHub\instance_mosaicking\jupyter_notebooks\synthetic_large_image_tutorial_output\workflow\merged_instances.json
Pixel polygon JSON: C:\Users\zhiang\Documents\GitHub\instance_mosaicking\jupyter_notebooks\synthetic_large_image_tutorial_output\workflow\merged_pixel_polygons.json
Instance mask PNG: C:\Users\zhiang\Documents\GitHub\instance_mosaicking\jupyter_notebooks\synthetic_large_image_tutorial_output\workflow\merged_instance_mask.png
Overlay PNG: C:\Users\zhiang\Documents\GitHub\instance_mosaicking\jupyter_notebooks\synthetic_large_image_tutorial_output\workflow\merged_overlay.png


In [6]:
metrics = evaluate(annotation_path, merged_path, IoU_threshold=0.5)

print(json.dumps(metrics, indent=2))
assert metrics["TP"] == ellipse_count
assert metrics["FP"] == 0
assert metrics["FN"] == 0
assert metrics["precision"] == 1.0
assert metrics["recall"] == 1.0

{
  "TP": 12,
  "FP": 0,
  "FN": 0,
  "precision": 1.0,
  "recall": 1.0,
  "accuracy": 1.0,
  "matches": [
    {
      "groundtruth_id": 1,
      "merged_id": 1,
      "iou": 0.7746529013883945
    },
    {
      "groundtruth_id": 2,
      "merged_id": 2,
      "iou": 0.7702041990221455
    },
    {
      "groundtruth_id": 3,
      "merged_id": 3,
      "iou": 0.7666169895678092
    },
    {
      "groundtruth_id": 4,
      "merged_id": 4,
      "iou": 0.7680473372781065
    },
    {
      "groundtruth_id": 9,
      "merged_id": 5,
      "iou": 0.7813846595805803
    },
    {
      "groundtruth_id": 10,
      "merged_id": 6,
      "iou": 0.7600107497984413
    },
    {
      "groundtruth_id": 5,
      "merged_id": 7,
      "iou": 0.7598230549050221
    },
    {
      "groundtruth_id": 7,
      "merged_id": 8,
      "iou": 0.7601410934744268
    },
    {
      "groundtruth_id": 11,
      "merged_id": 9,
      "iou": 0.7691025284801334
    },
    {
      "groundtruth_id": 6,
      "merge

## Instance-label mask annotation

The same fixture also writes an instance-label mask. Pixel value `0` is background and each nonzero value is one ellipse instance.

In [7]:
mask_annotation_path = Path(fixture["mask_path"])
mask_workflow_dir = output_root / "mask_workflow"

mask_workflow = Workflow(
    image_path=image_path,
    annotation_path=mask_annotation_path,
    annotation_format="mask",
    save_dir=mask_workflow_dir,
    tile_size=160,
    overlap=40,
    keep_empty_tiles=True,
    detection_threshold=0.0,
    iou_threshold=0.4,
)

mask_metadata = mask_workflow.split()
mask_pickle_paths = mask_workflow.save_training_pickles()
mask_workflow.register_predictions(mask_workflow_dir)

mask_merged_path = mask_workflow_dir / "merged_instances.json"
mask_merged = json.loads(mask_merged_path.read_text(encoding="utf-8"))
mask_metrics = evaluate(annotation_path, mask_merged_path, IoU_threshold=0.5)

print(f"Mask annotation: {mask_annotation_path}")
print(f"Mask tile count: {len(mask_metadata['tiles'])}")
print(f"Mask pickle count: {len(mask_pickle_paths)}")
print(f"Mask merged instances: {len(mask_merged['annotations'])}")
print(json.dumps(mask_metrics, indent=2))

assert (mask_workflow_dir / "tile_masks" / "0_0.png").exists()
assert len(mask_metadata["tiles"]) == 9
assert len(mask_pickle_paths) == 9
assert len(mask_merged["annotations"]) == ellipse_count
assert mask_metrics["TP"] == ellipse_count
assert mask_metrics["FP"] == 0
assert mask_metrics["FN"] == 0

Mask annotation: C:\Users\zhiang\Documents\GitHub\instance_mosaicking\jupyter_notebooks\synthetic_large_image_tutorial_output\input\large_image_fixture_mask.png
Mask tile count: 9
Mask pickle count: 9
Mask merged instances: 12
{
  "TP": 12,
  "FP": 0,
  "FN": 0,
  "precision": 1.0,
  "recall": 1.0,
  "accuracy": 1.0,
  "matches": [
    {
      "groundtruth_id": 1,
      "merged_id": 1,
      "iou": 0.7746529013883945
    },
    {
      "groundtruth_id": 2,
      "merged_id": 2,
      "iou": 0.7702041990221455
    },
    {
      "groundtruth_id": 3,
      "merged_id": 3,
      "iou": 0.7666169895678092
    },
    {
      "groundtruth_id": 4,
      "merged_id": 4,
      "iou": 0.7680473372781065
    },
    {
      "groundtruth_id": 9,
      "merged_id": 5,
      "iou": 0.7813846595805803
    },
    {
      "groundtruth_id": 10,
      "merged_id": 6,
      "iou": 0.7600107497984413
    },
    {
      "groundtruth_id": 5,
      "merged_id": 7,
      "iou": 0.7598230549050221
    },
    {
 